# 🛒 E-Commerce Exploratory Data Analysis

Interactive exploration of the **Olist Brazilian E-Commerce** dataset (2016–2018).

**Sections:**
1. Data Overview & Shape
2. Missing Values Analysis
3. Revenue Distribution
4. Customer Geography Breakdown
5. Review Score Distribution

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import plotly.express as px
from src.data_loader import EcommerceDataLoader

loader = EcommerceDataLoader(db_path="../data/ecommerce.db")
print("Ready!")

## 1. Data Overview & Shape

In [ ]:
tables = ["orders", "items", "products", "customers", "payments", "reviews"]

for table in tables:
    df = loader.query(f"SELECT * FROM {table} LIMIT 1")
    count = loader.query(f"SELECT COUNT(*) AS cnt FROM {table}")["cnt"].iloc[0]
    print(f"{table:12s} → {count:>9,} rows, {len(df.columns):>2} columns")
    display(df.head())

## 2. Missing Values Analysis

In [ ]:
for table in tables:
    df = loader.query(f"SELECT * FROM {table}")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n--- {table} ---")
        for col, val in missing.items():
            pct = val / len(df) * 100
            print(f"  {col}: {val:,} missing ({pct:.1f}%)")
    else:
        print(f"--- {table} --- No missing values ✅")

## 3. Revenue Distribution

In [ ]:
payments = loader.query("SELECT payment_value FROM payments")

fig = px.histogram(
    payments,
    x="payment_value",
    nbins=100,
    title="Payment Value Distribution",
    labels={"payment_value": "Payment Value (R$)", "count": "Frequency"},
    color_discrete_sequence=["#4C9BE8"],
)
fig.update_layout(template="plotly_white")
fig.show()

print(f"Mean:   R$ {payments['payment_value'].mean():,.2f}")
print(f"Median: R$ {payments['payment_value'].median():,.2f}")
print(f"Max:    R$ {payments['payment_value'].max():,.2f}")

## 4. Customer Geography Breakdown

In [ ]:
geo = loader.query("""
    SELECT customer_state, COUNT(*) AS customer_count
    FROM customers
    GROUP BY customer_state
    ORDER BY customer_count DESC
    LIMIT 15
""")

fig = px.bar(
    geo,
    x="customer_state",
    y="customer_count",
    title="Top 15 States by Customer Count",
    labels={"customer_state": "State", "customer_count": "Customers"},
    color="customer_count",
    color_continuous_scale="Blues",
)
fig.update_layout(template="plotly_white")
fig.show()

## 5. Review Score Distribution

In [ ]:
reviews = loader.query("""
    SELECT review_score, COUNT(*) AS review_count
    FROM reviews
    GROUP BY review_score
    ORDER BY review_score
""")

fig = px.bar(
    reviews,
    x="review_score",
    y="review_count",
    title="Review Score Distribution",
    labels={"review_score": "Score (1-5)", "review_count": "Number of Reviews"},
    color="review_score",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(template="plotly_white")
fig.show()

avg = reviews["review_score"].mean()
print(f"\nAverage review score: {avg:.2f} / 5")